# HW7 — GAIL: Generative Adversarial Imitation Learning

**A deep-dive homework on a single algorithm.** You will build GAIL (Ho & Ermon, 2016) completely from scratch: the RL "generator" (PPO with GAE), the discriminator (with one-sided label smoothing and a gradient penalty), three variants of the adversarial surrogate reward, the full training loop, and a set of ablations and visualizations that expose *why* the algorithm behaves the way it does.

**Environment:** `CartPole-v1`. Small on purpose — every experiment in this notebook runs on a laptop / Kaggle / Colab **CPU**. Total runtime of a correct notebook is roughly **15–20 minutes**; per-cell estimates are given where it matters.

**Grading:** 14 code TODOs (80 pts) + 5 written questions (20 pts) = 100 pts. Most TODOs are followed by `assert`-based sanity checks — they must pass, but passing them does not guarantee full credit.

**Rules:** don't modify the given code or the seeds; don't import anything beyond the setup cell; your GAIL learner must never train on the environment reward (it is logged for *monitoring only*).

**References** (you will want the first two open while working):
- Ho & Ermon, *Generative Adversarial Imitation Learning*, NeurIPS 2016 — [arXiv:1606.03476](https://arxiv.org/abs/1606.03476)
- Kostrikov et al., *Discriminator-Actor-Critic: Addressing Sample Inefficiency and Reward Bias in Adversarial Imitation Learning*, ICLR 2019 — [arXiv:1809.02925](https://arxiv.org/abs/1809.02925) (reward bias — Part 5)
- Fu, Luo, Levine, *AIRL*, 2018 — [arXiv:1710.11248](https://arxiv.org/abs/1710.11248) (the third reward variant)
- Gulrajani et al., *Improved Training of Wasserstein GANs*, 2017 — [arXiv:1704.00028](https://arxiv.org/abs/1704.00028) (gradient penalty)
- Schulman et al., [PPO](https://arxiv.org/abs/1707.06347) and [GAE](https://arxiv.org/abs/1506.02438)
- Reference implementations to *consult, not copy*: [`HumanCompatibleAI/imitation`](https://github.com/HumanCompatibleAI/imitation), [`hcnoh/gail-pytorch`](https://github.com/hcnoh/gail-pytorch)

## 0 · Setup

Dependencies: `torch`, `gymnasium`, `numpy`, `matplotlib` (`pip install torch gymnasium numpy matplotlib`).

In [ ]:
import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

ENV_ID = "CartPole-v1"
_probe = gym.make(ENV_ID)
OBS_DIM = _probe.observation_space.shape[0]   # 4: [cart pos, cart vel, pole angle, pole angular vel]
N_ACTIONS = _probe.action_space.n             # 2: push left / push right
_probe.close()
print(f"obs_dim={OBS_DIM}, n_actions={N_ACTIONS}")


def mlp(in_dim, out_dim, hidden=(64, 64), act=nn.Tanh):
    """Small fully-connected network used by every model in this homework."""
    layers, d = [], in_dim
    for h in hidden:
        layers += [nn.Linear(d, h), act()]
        d = h
    layers.append(nn.Linear(d, out_dim))
    return nn.Sequential(*layers)

## 1 · The generator: on-policy RL machinery

GAIL is a *sandwich*: an RL algorithm in which only the reward has been swapped out. Before touching anything adversarial you must build that RL algorithm — here, **PPO with GAE**. Two reasons it is not "given": (i) every design decision you make here (bootstrapping, advantage normalization, clipping) reappears inside GAIL, and a bug here shows up as "GAIL doesn't work" — you need to *own* this code to debug the adversarial part; (ii) the expert you will imitate is trained with this very code, so Part 2 doubles as your unit test.

Given: the `ActorCritic` container — a categorical policy $\pi_\theta(a\mid s)$ and a value head $V_\phi(s)$.

In [ ]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.actor = mlp(obs_dim, n_actions)   # logits
        self.critic = mlp(obs_dim, 1)          # V(s)

    def dist(self, obs):
        return torch.distributions.Categorical(logits=self.actor(obs))

    def value(self, obs):
        return self.critic(obs).squeeze(-1)

    @torch.no_grad()
    def step(self, obs):
        d = self.dist(obs)
        a = d.sample()
        return a, d.log_prob(a), self.value(obs)

    @torch.no_grad()
    def act_greedy(self, obs):
        return self.actor(obs).argmax(-1)


def greedy_fn(ac):
    """Wrap an ActorCritic into a plain obs -> greedy int action function."""
    def f(obs):
        return int(ac.act_greedy(torch.as_tensor(obs, dtype=torch.float32)).item())
    return f

### TODO 1 — deterministic evaluation *(4 pts)*

Every score reported in this notebook comes from this one function, so make it fair: use `env.reset(seed=seed + episode_index)` so that **every** policy is evaluated from the same set of start states and numbers are comparable across experiments.

In [ ]:
@torch.no_grad()
def evaluate(policy, n_episodes=10, seed=1234):
    """policy: function mapping np obs -> int action. Returns mean episodic env return."""
    # ========== YOUR CODE HERE (~11 lines of solution) ==========
    env = gym.make(ENV_ID)
    returns = []
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=seed + episode)
        ep_ret = 0.0
        while True:
            action = int(policy(obs))
            obs, reward, terminated, truncated, _ = env.step(action)
            ep_ret += float(reward)
            if terminated or truncated:
                break
        returns.append(ep_ret)
    env.close()
    return float(np.mean(returns))
    # ==========================================

_r = evaluate(lambda o: int(rng.integers(N_ACTIONS)), n_episodes=5)
assert 5 <= _r <= 60, f"random policy should score ~15-30 on CartPole, got {_r}"
print(f"random policy: {_r:.1f}  (sanity check passed)")

### TODO 2 — rollout collection *(8 pts)*

Collect **exactly `n_steps` transitions** with the sampling policy, resetting whenever an episode ends (a rollout therefore crosses episode boundaries). This is the data pump for both expert training and GAIL.

The hard part is the **time-limit truncation**: CartPole episodes are cut at 500 steps by a `TimeLimit` wrapper. A truncated episode did *not* fail — treating truncation as death teaches the value function that step 499 is worthless, which caps learning. Gymnasium separates `terminated` (pole fell — real absorbing state, future value $0$) from `truncated` (timeout — the episode *would have continued*). When a step is truncated but **not** terminated, store $V(s_{t+1})$ in `boot[t]` so GAE can bootstrap *through* the cut; otherwise `boot[t] = 0`.

In [ ]:
def rollout(env, ac, n_steps, obs0):
    """Collect exactly n_steps transitions with pi_theta (sampling), resetting at episode ends.

    Returns
    -------
    data : dict of flat np arrays of length n_steps
        obs, act, rew (ENV reward), done (1.0 where an episode ended after the step),
        logp, val, boot (value bootstrap for time-limit truncations, else 0), and
        ep_step (the zero-based step index inside the current episode).
    ep_rets : list of COMPLETE env returns for episodes that finished during this rollout
    obs : final observation (pass back in as obs0 of the next call)
    last_val : V(obs) for bootstrapping the unfinished tail episode

    The ongoing episode return and episode-local step are stored on `env`, so an
    unfinished tail is continued correctly by the next call.
    """
    obs_buf = np.zeros((n_steps, OBS_DIM), dtype=np.float32)
    act_buf = np.zeros(n_steps, dtype=np.int64)
    rew_buf = np.zeros(n_steps, dtype=np.float32)
    done_buf = np.zeros(n_steps, dtype=np.float32)
    logp_buf = np.zeros(n_steps, dtype=np.float32)
    val_buf = np.zeros(n_steps, dtype=np.float32)
    boot_buf = np.zeros(n_steps, dtype=np.float32)
    ep_step_buf = np.zeros(n_steps, dtype=np.int64)

    # ========== YOUR CODE HERE (~22 lines of solution) ==========
    obs = np.asarray(obs0, dtype=np.float32)
    ep_rets = []
    ep_ret = float(getattr(env, "_hw7_ep_ret", 0.0))
    ep_step = int(getattr(env, "_hw7_ep_step", 0))

    for t in range(n_steps):
        obs_t = torch.as_tensor(obs, dtype=torch.float32)
        action, logp, value = ac.step(obs_t)
        next_obs, reward, terminated, truncated, _ = env.step(int(action.item()))
        ended = bool(terminated or truncated)

        obs_buf[t] = obs
        act_buf[t] = int(action.item())
        rew_buf[t] = float(reward)
        done_buf[t] = float(ended)
        logp_buf[t] = float(logp.item())
        val_buf[t] = float(value.item())
        ep_step_buf[t] = ep_step

        # True termination has continuation value zero. A time-limit cut is not
        # absorbing, so bootstrap from the observation produced by that action.
        if truncated and not terminated:
            boot_buf[t] = float(ac.value(
                torch.as_tensor(next_obs, dtype=torch.float32)
            ).item())

        ep_ret += float(reward)
        ep_step += 1

        if ended:
            ep_rets.append(ep_ret)
            ep_ret = 0.0
            ep_step = 0
            obs, _ = env.reset()
            obs = np.asarray(obs, dtype=np.float32)
        else:
            obs = np.asarray(next_obs, dtype=np.float32)

    # Preserve the unfinished episode across fixed-length rollout calls.
    env._hw7_ep_ret = ep_ret
    env._hw7_ep_step = ep_step
    # ==========================================

    last_val = ac.value(torch.as_tensor(obs, dtype=torch.float32)).item()
    data = dict(obs=obs_buf, act=act_buf, rew=rew_buf, done=done_buf,
                logp=logp_buf, val=val_buf, boot=boot_buf,
                ep_step=ep_step_buf)
    return data, ep_rets, obs, last_val

# sanity check
_env = gym.make(ENV_ID)
_o, _ = _env.reset(seed=0)
_data, _eps, _, _ = rollout(_env, ActorCritic(OBS_DIM, N_ACTIONS), 300, _o)
_env.close()
assert _data["obs"].shape == (300, OBS_DIM) and _data["act"].dtype == np.int64
assert _data["ep_step"].shape == (300,) and _data["ep_step"].dtype == np.int64
assert _data["done"].sum() == len(_eps), "one `done` flag per finished episode"
assert len(_eps) >= 3, "an untrained policy should finish several episodes in 300 steps"
_tail = int(_data["done"].nonzero()[0][-1]) + 1   # index right after the last finished episode
assert np.isclose(sum(_eps), _data["rew"][:_tail].sum()), \
    "episode returns must equal the summed rewards of the finished episodes"
print(f"rollout ok: {len(_eps)} episodes finished, mean return {np.mean(_eps):.1f}")

### TODO 3 — Generalized Advantage Estimation *(8 pts)*

Implement GAE($\gamma,\lambda$) over the flat arrays, running the recursion **backwards**:
$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t), \qquad \hat A_t = \delta_t + \gamma\lambda\,(1 - d_t)\,\hat A_{t+1},$$
with three boundary cases you must get right: (i) at a **terminated** step the next value is $0$ and the accumulator resets; (ii) at a **truncated** step the next value is `boot[t]` and the accumulator still resets (the next buffer entry belongs to a different episode!); (iii) at the buffer's end the next value is `last_val`. Return `adv` and the value targets `ret = adv + val`.

In [ ]:
def compute_gae(rew, val, done, boot, last_val, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation over the flat rollout arrays."""
    n = len(rew)
    adv = np.zeros(n, dtype=np.float32)
    # ========== YOUR CODE HERE (~12 lines of solution) ==========
    gae = 0.0
    for t in range(n - 1, -1, -1):
        if done[t]:
            next_value = float(boot[t])  # zero at termination, V(next_obs) at truncation
            continuation = 0.0          # never cross into the reset episode
        elif t == n - 1:
            next_value = float(last_val)
            continuation = 1.0
        else:
            next_value = float(val[t + 1])
            continuation = 1.0
        delta = float(rew[t]) + gamma * next_value - float(val[t])
        gae = delta + gamma * lam * continuation * gae
        adv[t] = gae
    # ==========================================
    ret = adv + val
    return adv, ret

# sanity: constant reward 1, V=0, one episode terminating at the last step
_a, _r = compute_gae(np.ones(3, np.float32), np.zeros(3, np.float32),
                     np.array([0, 0, 1], np.float32), np.zeros(3, np.float32),
                     last_val=0.0, gamma=0.5, lam=1.0)
assert np.allclose(_a, [1.75, 1.5, 1.0]), f"GAE recursion wrong: {_a}"
# truncation must bootstrap: boot value 2 at the cut raises the advantage there by gamma*2
_a2, _ = compute_gae(np.ones(3, np.float32), np.zeros(3, np.float32),
                     np.array([0, 0, 1], np.float32), np.array([0, 0, 2], np.float32),
                     last_val=0.0, gamma=0.5, lam=1.0)
assert np.allclose(_a2, [2.0, 2.0, 2.0]), f"truncation bootstrap wrong: {_a2}"
print("GAE sanity checks passed")

### TODO 4 — PPO update *(8 pts)*

One PPO update on a rollout: for `epochs` passes over shuffled minibatches, maximize the clipped surrogate with a value loss and an entropy bonus,
$$L = -\min\!\big(\rho_t \hat A_t,\; \mathrm{clip}(\rho_t, 1\!-\!\epsilon, 1\!+\!\epsilon)\, \hat A_t\big) + c_v\, \big(V_\phi(s_t) - \hat R_t\big)^2 - c_e\, H\big(\pi_\theta(\cdot\mid s_t)\big),$$
where $\rho_t = \exp(\log\pi_\theta(a_t\mid s_t) - \log\pi_{\theta_\text{old}}(a_t\mid s_t))$. Normalize advantages (mean 0, std 1) **once, over the whole batch**, and clip the global grad-norm to 0.5. The entropy bonus is not decoration: it is the $-\lambda H(\pi)$ term of the GAIL objective (Q4 asks about this).

In [ ]:
def ppo_update(ac, opt, data, adv, ret, clip=0.2, epochs=10, minibatch=256,
               vf_coef=0.5, ent_coef=0.01):
    """One PPO update (clipped surrogate + value loss + entropy bonus)."""
    obs = torch.as_tensor(data["obs"])
    act = torch.as_tensor(data["act"])
    logp_old = torch.as_tensor(data["logp"])
    # ========== YOUR CODE HERE (~18 lines of solution) ==========
    adv_t = torch.as_tensor(adv, dtype=torch.float32)
    ret_t = torch.as_tensor(ret, dtype=torch.float32)
    adv_t = (adv_t - adv_t.mean()) / (adv_t.std(unbiased=False) + 1e-8)

    n = len(obs)
    for _ in range(epochs):
        order = torch.randperm(n)
        for start in range(0, n, minibatch):
            idx = order[start:start + minibatch]
            dist = ac.dist(obs[idx])
            logp = dist.log_prob(act[idx])
            ratio = torch.exp(logp - logp_old[idx])
            unclipped = ratio * adv_t[idx]
            clipped = torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv_t[idx]
            policy_loss = -torch.minimum(unclipped, clipped).mean()
            value_loss = F.mse_loss(ac.value(obs[idx]), ret_t[idx])
            entropy = dist.entropy().mean()
            loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
            opt.step()
    # ==========================================

## 2 · The expert and its demonstrations

The cell below trains the expert **with your TODO 2–4 code** on the true reward (given — this is your integration test). It should converge to a greedy return $\ge 495$ within ~100 iterations (**~30–60 s**). If it plateaus below 400, the bug is in your rollout / GAE / PPO — fix it *now*, before GAIL can hide it.

In [ ]:
def train_expert(max_iters=100, n_steps=2048, seed=SEED, eval_target=495):
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=seed)
    ac = ActorCritic(OBS_DIM, N_ACTIONS)
    opt = torch.optim.Adam(ac.parameters(), lr=3e-4)
    for it in range(1, max_iters + 1):
        data, ep_rets, obs, last_val = rollout(env, ac, n_steps, obs)
        adv, ret = compute_gae(data["rew"], data["val"], data["done"], data["boot"], last_val)
        ppo_update(ac, opt, data, adv, ret)
        mean_ret = np.mean(ep_rets) if ep_rets else float("nan")
        if mean_ret > 400:
            g = evaluate(greedy_fn(ac), n_episodes=5)
            print(f"iter {it:3d} | rollout return {mean_ret:6.1f} | greedy eval {g:6.1f}")
            if g >= eval_target:
                break
        elif it % 5 == 0 or it == 1:
            print(f"iter {it:3d} | rollout return {mean_ret:6.1f}")
    env.close()
    return ac

t0 = time.time()
expert = train_expert()
print(f"\ntrained in {time.time()-t0:.0f}s")
expert_score = evaluate(greedy_fn(expert), n_episodes=20)
print(f"expert greedy return over 20 episodes: {expert_score:.1f} / 500")
assert expert_score >= 475, "expert did not converge — rerun this cell"

### TODO 5 — collect demonstrations, subsampled *(4 pts)*

Roll out the **greedy** expert for `n_episodes` episodes and record $(s, a)$ pairs — but keep only **every `subsample`-th** transition of each episode. Consecutive CartPole states are nearly identical; Ho & Ermon subsampled demonstrations aggressively (their headline results use as few as ~50–1000 state–action pairs per task) both to de-correlate the expert dataset and to show GAIL's expert-sample efficiency, which you will measure yourself in TODO 12. Also return the (full, un-subsampled) episodic returns so we can verify the demos really come from an expert.

In [ ]:
def collect_demos(expert, n_episodes, subsample=20, seed=100):
    """Returns (S, A, ep_rets): float32 [N, OBS_DIM], int64 [N], and full episodic returns.
    Keeps transitions t = 0, subsample, 2*subsample, ... of each episode."""
    # ========== YOUR CODE HERE (~16 lines of solution) ==========
    env = gym.make(ENV_ID)
    states, actions, ep_rets = [], [], []
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=seed + episode)
        ep_ret = 0.0
        t = 0
        while True:
            action = int(expert.act_greedy(
                torch.as_tensor(obs, dtype=torch.float32)
            ).item())
            if t % subsample == 0:
                states.append(np.asarray(obs, dtype=np.float32).copy())
                actions.append(action)
            obs, reward, terminated, truncated, _ = env.step(action)
            ep_ret += float(reward)
            t += 1
            if terminated or truncated:
                break
        ep_rets.append(ep_ret)
    env.close()
    return (np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64), ep_rets)
    # ==========================================

demo_S, demo_A, demo_rets = collect_demos(expert, n_episodes=25, subsample=20)
print(f"{len(demo_S)} expert (s,a) pairs from {len(demo_rets)} episodes, "
      f"mean demo return {np.mean(demo_rets):.1f}")
assert demo_S.shape == (len(demo_A), OBS_DIM) and demo_S.dtype == np.float32
assert np.mean(demo_rets) >= 475, "demos must come from a working expert"
assert len(demo_S) <= 25 * 25 + 25, "did you forget to subsample?"

## 3 · GAIL: theory, discriminator, surrogate reward

### 3.1 Theory in one page

Define the (discounted) **occupancy measure** of a policy,
$$\rho_\pi(s, a) \;=\; \pi(a \mid s) \sum_{t=0}^{\infty} \gamma^t \Pr(s_t = s \mid \pi),$$
the distribution over state–action pairs the policy generates. There is a one-to-one correspondence between policies and valid occupancy measures, so *"imitate the expert"* can be recast as *"match $\rho_\pi$ to $\rho_{\pi^E}$"*.

Ho & Ermon showed that (entropy-regularized) **IRL followed by RL** — infer a cost, then plan against it — collapses, for a suitable cost regularizer $\psi$, into a single saddle-point problem over occupancy measures:
$$\min_\pi \; \psi^*(\rho_\pi - \rho_{\pi^E}) \;-\; \lambda H(\pi).$$
For their particular $\psi$ the divergence $\psi^*$ becomes the **Jensen–Shannon divergence**, estimated exactly the way GANs estimate it — with a learned binary classifier $D_\omega(s,a) \approx \Pr(\text{expert} \mid s,a)$:

$$\max_\omega \;\; \mathbb{E}_{\rho_{\pi^E}}\big[\log D_\omega(s,a)\big] + \mathbb{E}_{\rho_\pi}\big[\log(1 - D_\omega(s,a))\big], \qquad \min_\pi \;\; \mathbb{E}_{\rho_\pi}\big[\log(1 - D_\omega(s,a))\big] - \lambda H(\pi).$$

The **generator is the policy**, trained with any RL algorithm — your Part 1 PPO — on a **surrogate reward** derived from $D$. Unlike a GAN generator, the policy cannot backprop through $D$ (the environment sits in between, Q4); the discriminator's opinion reaches the policy *only through scalar rewards*.

> **Algorithm (GAIL).** Repeat: (1) roll out $\pi_\theta$, *discarding env rewards*; (2) a few SGD steps on $D_\omega$ — expert batch vs. rollout batch; (3) relabel the rollout with the surrogate reward; (4) one PPO update.

### 3.2 The surrogate reward is a design choice — and it leaks bias

Three forms are used in the literature, all functions of the discriminator **logit** $x$ (where $D = \sigma(x)$):

| name | formula | in terms of the logit | sign | used by |
|---|---|---|---|---|
| `gail` | $-\log(1-D)$ | $\operatorname{softplus}(x)$ | always $> 0$ | most GAIL implementations |
| `logd` | $\log D$ | $-\operatorname{softplus}(-x)$ | always $< 0$ | original GAIL paper's cost |
| `airl` | $\log D - \log(1-D)$ | $x$ | either | AIRL, DAC |

A reward that is *always positive* pays the agent for merely staying alive; one that is *always negative* pays it to end episodes quickly. Kostrikov et al. (2019) call this **reward bias**. The logit reward can take either sign — but "unbiased in form" is not the same as "unbiased in effect": what matters is the sign of the reward *in the region of state–action space the current policy actually visits*. You will observe all of this directly in TODO 11.

### TODO 6 — the discriminator *(4 pts)*

$D_\omega(s,a)$ scores state–action pairs and returns the raw **logit** (no sigmoid — the BCE-with-logits loss and all three rewards are computed stably from the logit). With discrete actions, concatenate the state with a **one-hot** action encoding. One subtlety: the gradient penalty (TODO 7) will need to evaluate $D$ at *interpolated*, non-integer action encodings — so `forward` must also accept an already-one-hot **float** action tensor.

In [ ]:
class Discriminator(nn.Module):
    """D(s, a) -> raw logit;  sigmoid(logit) = P(expert | s, a)."""
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.n_actions = n_actions
        # ========== YOUR CODE HERE (~1 lines of solution) ==========
        self.net = mlp(obs_dim + n_actions, 1)
        # ==========================================

    def forward(self, obs, act):
        """obs: float [B, OBS_DIM]. act: int64 [B] (one-hot it) OR float [B, n_actions]
        (already one-hot / interpolated — use as is). Returns float [B] of logits."""
        # ========== YOUR CODE HERE (~4 lines of solution) ==========
        if not torch.is_floating_point(act):
            act = F.one_hot(act.long(), num_classes=self.n_actions).to(obs.dtype)
        else:
            act = act.to(dtype=obs.dtype)
        return self.net(torch.cat([obs, act], dim=-1)).squeeze(-1)
        # ==========================================

# shape checks: int actions and soft one-hot actions must both work
_d = Discriminator(OBS_DIM, N_ACTIONS)
assert _d(torch.randn(7, OBS_DIM), torch.randint(0, N_ACTIONS, (7,))).shape == (7,)
assert _d(torch.randn(7, OBS_DIM), torch.rand(7, N_ACTIONS)).shape == (7,)
print("discriminator shape checks passed")

### TODO 7 — gradient penalty *(6 pts)*

An unconstrained discriminator on low-dimensional CartPole becomes near-perfect within a few iterations; its logits saturate, the surrogate reward flattens, and the policy's learning signal dies (Q3). The standard fix (Gulrajani et al. 2017; used for GAIL by Kostrikov et al. 2019) penalizes the discriminator's gradient on **interpolates** between expert and policy samples:
$$\text{GP} = \mathbb{E}_{\hat x}\big[(\lVert \nabla_{\hat x} D(\hat x) \rVert_2 - 1)^2\big], \qquad \hat x = \epsilon\, x_{\text{expert}} + (1-\epsilon)\, x_{\text{policy}},\; \epsilon \sim U[0,1].$$

Implementation notes: interpolate the observation *and* the one-hot action encoding with the **same** per-sample $\epsilon$; mark both interpolates `requires_grad_(True)`; get the gradients with `torch.autograd.grad(logits.sum(), [x_obs, x_act], create_graph=True)` (`create_graph=True` so the penalty itself can be backpropagated; the `.sum()` trick is valid because sample $i$'s logit depends only on sample $i$'s input); concatenate the two gradients before taking the norm.

In [ ]:
def gradient_penalty(disc, obs_e, act_e, obs_p, act_p):
    """One-centered gradient penalty on expert/policy interpolates.
    obs_*: float [B, OBS_DIM]; act_*: int64 [B]. Returns a scalar tensor."""
    # ========== YOUR CODE HERE (~10 lines of solution) ==========
    act_e_oh = F.one_hot(act_e.long(), num_classes=disc.n_actions).to(obs_e.dtype)
    act_p_oh = F.one_hot(act_p.long(), num_classes=disc.n_actions).to(obs_p.dtype)
    eps = torch.rand(len(obs_e), 1, device=obs_e.device, dtype=obs_e.dtype)
    obs_hat = (eps * obs_e + (1.0 - eps) * obs_p).requires_grad_(True)
    act_hat = (eps * act_e_oh + (1.0 - eps) * act_p_oh).requires_grad_(True)
    logits = disc(obs_hat, act_hat)
    grad_obs, grad_act = torch.autograd.grad(
        logits.sum(), [obs_hat, act_hat], create_graph=True
    )
    full_grad = torch.cat([grad_obs, grad_act], dim=-1)
    return ((full_grad.norm(2, dim=-1) - 1.0) ** 2).mean()
    # ==========================================

# sanity: scalar, differentiable, non-negative
_gp = gradient_penalty(_d, torch.randn(16, OBS_DIM), torch.randint(0, N_ACTIONS, (16,)),
                       torch.randn(16, OBS_DIM), torch.randint(0, N_ACTIONS, (16,)))
assert _gp.dim() == 0 and _gp.item() >= 0 and _gp.requires_grad
_gp.backward()  # must not crash: create_graph=True was required
print(f"gradient penalty ok: {_gp.item():.3f}")

### TODO 8 — discriminator update *(6 pts)*

`d_steps` minibatch steps of binary cross-entropy **on logits** (`F.binary_cross_entropy_with_logits`): expert batch → target $1 - $ `smooth` (one-sided label smoothing — never let the discriminator get *fully* confident about the expert class), policy batch → target $0$, plus `gp_coef` times your gradient penalty. Keep `d_steps` small: the discriminator only needs to stay *slightly ahead* of the policy (Q3).

Also return diagnostics — you will plot them in Part 4: the last BCE loss, and the classification accuracies on the last expert / policy batches (fraction of expert logits $> 0$, fraction of policy logits $< 0$).

In [ ]:
def update_discriminator(disc, d_opt, expert_S, expert_A, policy_S, policy_A,
                         d_steps=4, batch=256, smooth=0.1, gp_coef=1.0):
    """Returns dict(loss=last BCE, acc_expert=..., acc_policy=...)."""
    eS, eA = torch.as_tensor(expert_S), torch.as_tensor(expert_A)
    pS, pA = torch.as_tensor(policy_S), torch.as_tensor(policy_A)
    # ========== YOUR CODE HERE (~16 lines of solution) ==========
    stats = None
    for _ in range(d_steps):
        ei = torch.randint(len(eS), (batch,))
        pi = torch.randint(len(pS), (batch,))
        obs_e, act_e = eS[ei], eA[ei]
        obs_p, act_p = pS[pi], pA[pi]

        logits_e = disc(obs_e, act_e)
        logits_p = disc(obs_p, act_p)
        target_e = torch.full_like(logits_e, 1.0 - smooth)
        target_p = torch.zeros_like(logits_p)
        bce = 0.5 * (
            F.binary_cross_entropy_with_logits(logits_e, target_e) +
            F.binary_cross_entropy_with_logits(logits_p, target_p)
        )
        gp = gradient_penalty(disc, obs_e, act_e, obs_p, act_p)
        loss = bce + gp_coef * gp

        d_opt.zero_grad()
        loss.backward()
        d_opt.step()
        stats = dict(
            loss=float(bce.detach().item()),
            acc_expert=float((logits_e.detach() > 0).float().mean().item()),
            acc_policy=float((logits_p.detach() < 0).float().mean().item()),
        )
    return stats
    # ==========================================

# sanity: on trivially separable data the discriminator must learn within a few calls
_d2 = Discriminator(OBS_DIM, N_ACTIONS)
_o2 = torch.optim.Adam(_d2.parameters(), lr=1e-3)
_Se = np.full((512, OBS_DIM), +1.0, np.float32); _Ae = np.zeros(512, np.int64)
_Sp = np.full((512, OBS_DIM), -1.0, np.float32); _Ap = np.ones(512, np.int64)
for _ in range(20):
    _st = update_discriminator(_d2, _o2, _Se, _Ae, _Sp, _Ap)
assert set(_st) >= {"loss", "acc_expert", "acc_policy"}, "return the diagnostics dict"
assert _st["acc_expert"] > 0.9 and _st["acc_policy"] > 0.9, f"did not separate: {_st}"
print(f"discriminator update ok: {_st}")

### TODO 9 — the surrogate reward, three ways *(6 pts)*

Implement all three reward variants of §3.2 **numerically stably, straight from the logit** — never materialize $D$ itself: `-log(1 - sigmoid(x))` overflows to `inf` for a confidently-expert logit. With $D = \sigma(x)$ and $1 - \sigma(x) = \sigma(-x)$:
$$r_\text{gail} = -\log\sigma(-x) = \operatorname{softplus}(x), \qquad r_\text{logd} = \log\sigma(x) = -\operatorname{softplus}(-x), \qquad r_\text{airl} = \log\tfrac{D}{1-D} = x.$$
Note the identity $r_\text{airl} = r_\text{gail} + r_\text{logd}$ — the sanity check below uses it.

In [ ]:
@torch.no_grad()
def gail_reward(disc, obs_np, act_np, kind="gail"):
    """Surrogate reward from the discriminator logit. Returns float32 [N] array.
    kind: 'gail' = -log(1-D) | 'logd' = log D | 'airl' = log D - log(1-D)."""
    logits = disc(torch.as_tensor(obs_np), torch.as_tensor(act_np))
    # ========== YOUR CODE HERE (~8 lines of solution) ==========
    if kind == "gail":
        reward = F.softplus(logits)
    elif kind == "logd":
        reward = -F.softplus(-logits)
    elif kind == "airl":
        reward = logits
    else:
        raise ValueError(f"unknown reward kind: {kind}")
    return reward.cpu().numpy().astype(np.float32, copy=False)
    # ==========================================

# sanity: signs, stability at extreme logits, and the airl = gail + logd identity
_S9 = np.random.randn(64, OBS_DIM).astype(np.float32)
_A9 = np.random.randint(0, N_ACTIONS, 64).astype(np.int64)
_rg = gail_reward(_d, _S9, _A9, "gail"); _rl = gail_reward(_d, _S9, _A9, "logd")
_ra = gail_reward(_d, _S9, _A9, "airl")
assert (_rg > 0).all() and (_rl < 0).all(), "gail must be >0, logd must be <0"
assert np.allclose(_ra, _rg + _rl, atol=1e-5), "identity airl = gail + logd violated"
with torch.no_grad():
    _d.net[-1].bias.fill_(60.0)   # push logits to ±60: naive sigmoid math would inf/nan here
assert np.isfinite(gail_reward(_d, _S9, _A9, "gail")).all()
assert np.isfinite(gail_reward(_d, _S9, _A9, "logd")).all()
with torch.no_grad():
    _d.net[-1].bias.zero_()
print("surrogate reward checks passed")

## 4 · The GAIL training loop

### TODO 10 — put it together *(10 pts)*

Wire everything up, reusing `rollout`, `compute_gae`, and `ppo_update` from Part 1 **unchanged**. Each iteration:

1. `rollout` the current policy — the returned `rew` is the env reward; append its per-episode mean to the history as a *monitoring* signal, but the learner must never train on it;
2. `update_discriminator` on the expert data vs. this rollout;
3. **overwrite** `data["rew"]` with `gail_reward(..., kind=kind)` — the *only* line where imitation enters the RL update;
4. `compute_gae` + `ppo_update`, exactly as in expert training.

Log per iteration into `hist` (a dict of lists): `env_ret` (NaN if no episode finished), `d_loss`, `acc_expert`, `acc_policy`, `surr` (mean surrogate reward of the rollout), and `entropy` (mean policy entropy over the rollout states, under `torch.no_grad()`).

5. One last, essential ingredient: **adversarial training does not converge monotonically.** The reward function is *moving* — a policy that was near-expert at iteration 40 can be visibly worse at iteration 60 as the discriminator shifts under it. Nobody deploys "whatever the last iteration produced"; you keep the best snapshot. Every `eval_every` iterations, run a greedy evaluation (`evaluate(greedy_fn(ac), n_episodes=5, seed=999)`), append `(it, score)` to `hist["eval"]`, and if it beats the best score so far, save `copy.deepcopy(ac.state_dict())`. If `target` is not `None`, stop early once the eval score reaches it. **After the loop, restore the best snapshot into `ac`.**

Expected: **~1–2 min** for 60 iterations, env return climbing toward 500 even though the learner never sees an env reward.

In [ ]:
def gail(expert_S, expert_A, kind="gail", n_iters=60, n_steps=2048, target=None,
         eval_every=5, seed=SEED, verbose=True):
    """Returns (ac, disc, hist). ac is the BEST evaluated snapshot, not the last iterate."""
    torch.manual_seed(seed)
    env = gym.make(ENV_ID)
    obs, _ = env.reset(seed=seed + 7)
    ac = ActorCritic(OBS_DIM, N_ACTIONS)
    opt = torch.optim.Adam(ac.parameters(), lr=3e-4)
    disc = Discriminator(OBS_DIM, N_ACTIONS)
    d_opt = torch.optim.Adam(disc.parameters(), lr=3e-4)
    hist = {k: [] for k in ["env_ret", "d_loss", "acc_expert", "acc_policy",
                            "surr", "entropy", "eval"]}
    # ========== YOUR CODE HERE (~32 lines of solution) ==========
    best_score = -np.inf
    best_state = None
    best_iter = None

    for it in range(1, n_iters + 1):
        data, ep_rets, obs, last_val = rollout(env, ac, n_steps, obs)

        # Equal class weighting is implemented inside update_discriminator:
        # one expert minibatch and one fresh on-policy minibatch per D step.
        d_stats = update_discriminator(
            disc, d_opt, expert_S, expert_A, data["obs"], data["act"]
        )

        # Freeze D conceptually: rewards are materialized as fixed NumPy scalars,
        # and PPO never receives a computational graph into the discriminator.
        surrogate = gail_reward(disc, data["obs"], data["act"], kind=kind)
        data["rew"] = surrogate
        adv, ret = compute_gae(
            data["rew"], data["val"], data["done"], data["boot"], last_val
        )
        ppo_update(ac, opt, data, adv, ret)

        with torch.no_grad():
            entropy = ac.dist(torch.as_tensor(data["obs"])).entropy().mean().item()
        hist["env_ret"].append(float(np.mean(ep_rets)) if ep_rets else float("nan"))
        hist["d_loss"].append(d_stats["loss"])
        hist["acc_expert"].append(d_stats["acc_expert"])
        hist["acc_policy"].append(d_stats["acc_policy"])
        hist["surr"].append(float(np.mean(surrogate)))
        hist["entropy"].append(float(entropy))

        if it % eval_every == 0:
            score = evaluate(greedy_fn(ac), n_episodes=5, seed=999)
            hist["eval"].append((it, score))
            if score > best_score:
                best_score = score
                best_state = copy.deepcopy(ac.state_dict())
                best_iter = it
            if verbose:
                print(f"iter {it:3d} | env {hist['env_ret'][-1]:6.1f} | "
                      f"D {d_stats['acc_expert']:.2f}/{d_stats['acc_policy']:.2f} | "
                      f"surr {hist['surr'][-1]:7.3f} | eval {score:6.1f}")
            if target is not None and score >= target:
                break

    # Record the last iterate BEFORE restoring the selected checkpoint.
    final_iter_score = evaluate(greedy_fn(ac), n_episodes=5, seed=999)
    if best_state is None:
        best_score = final_iter_score
        best_state = copy.deepcopy(ac.state_dict())
        best_iter = len(hist["env_ret"])
        hist["eval"].append((best_iter, best_score))

    hist["final_eval"] = float(final_iter_score)
    hist["best_eval"] = float(best_score)
    hist["best_iter"] = int(best_iter)
    ac.load_state_dict(best_state)
    # ==========================================
    env.close()
    return ac, disc, hist

t0 = time.time()
gail_ac, gail_disc, gail_hist = gail(demo_S, demo_A, kind="gail", n_iters=60)
print(f"\nGAIL trained in {time.time()-t0:.0f}s")
gail_score = evaluate(greedy_fn(gail_ac), n_episodes=20)
print(f"GAIL restored-best greedy eval over 20 episodes: {gail_score:.1f} / 500 "
      f"(expert: {expert_score:.1f})")
print(f"selection protocol (5 fixed episodes): final iterate "
      f"{gail_hist['final_eval']:.1f}; best checkpoint "
      f"{gail_hist['best_eval']:.1f} at iteration {gail_hist['best_iter']}")
assert gail_score >= 450, "GAIL under-performed — check TODOs 6-10, then rerun"

The four diagnostics below (given) are how practitioners read an adversarial run. Healthy GAIL: env return climbs *but keeps oscillating* — the reward is a moving target, which is exactly why your loop keeps the best snapshot (crosses) instead of the last iterate; discriminator accuracy hovers **near-but-above 0.5** (it stays slightly ahead — never pinned at 1.0, which would mean saturation, and never 0.5 from the start, which would mean no learning signal at all); the mean surrogate reward *rises* as the policy fools the discriminator; entropy decays as the policy commits.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
its = np.arange(1, len(gail_hist["env_ret"]) + 1)
axes[0, 0].plot(its, gail_hist["env_ret"], c="tab:blue", label="rollout (sampling)")
_ev = np.array(gail_hist["eval"])
axes[0, 0].plot(_ev[:, 0], _ev[:, 1], "x", c="tab:orange", label="greedy eval snapshots")
axes[0, 0].axhline(expert_score, ls="--", c="gray", label="expert")
axes[0, 0].set_title("env return (monitoring only)"); axes[0, 0].legend(fontsize=8)
axes[0, 1].plot(its, gail_hist["acc_expert"], label="on expert data")
axes[0, 1].plot(its, gail_hist["acc_policy"], label="on policy data")
axes[0, 1].axhline(0.5, ls=":", c="gray")
axes[0, 1].set_ylim(0, 1.05); axes[0, 1].set_title("discriminator accuracy"); axes[0, 1].legend()
axes[1, 0].plot(its, gail_hist["surr"], c="tab:green")
axes[1, 0].set_title("mean surrogate reward $-\\log(1-D)$")
axes[1, 1].plot(its, gail_hist["entropy"], c="tab:red")
axes[1, 1].set_title("policy entropy")
for ax in axes.flat:
    ax.set_xlabel("GAIL iteration")
plt.tight_layout(); plt.show()

## 5 · Analysis: ablations and visualizations

### TODO 11 — reward bias ablation *(4 pts)*

Run GAIL with the **other two** reward variants (`airl`, `logd`) for 60 iterations each, no early stopping (**~2 min each**), and plot all three env-return curves together — reuse `gail_hist` for the `gail` curve. Store each run's history in `reward_runs[kind]` and its final greedy eval in `reward_scores[kind]`. Before running, write down your prediction: which variant should suffer on CartPole, and in which direction? (Q2 asks you to explain what you see.)

In [ ]:
reward_runs = {"gail": gail_hist}
reward_scores = {"gail": gail_score}  # restored-best checkpoints, evaluated over 20 episodes
reward_final_scores = {"gail": gail_hist["final_eval"]}  # final iterates, 5-episode selection protocol
# ========== YOUR CODE HERE (~14 lines of solution) ==========
print("Prediction before running: the always-negative 'logd' reward should most strongly "
      "favor early termination on CartPole; the positive 'gail' reward should benefit from "
      "CartPole's survival objective, while the signed logit ('airl') should be intermediate.")
for kind in ["airl", "logd"]:
    ac_k, _, hist_k = gail(
        demo_S, demo_A, kind=kind, n_iters=60, target=None,
        seed=SEED, verbose=False
    )
    reward_runs[kind] = hist_k
    reward_scores[kind] = evaluate(greedy_fn(ac_k), n_episodes=20)
    reward_final_scores[kind] = hist_k["final_eval"]
    print(f"{kind:5s} restored-best eval (20 eps): {reward_scores[kind]:6.1f} | "
          f"final iterate (5 fixed eps): {hist_k['final_eval']:6.1f} | "
          f"best selected at iter {hist_k['best_iter']:2d}")

plt.figure(figsize=(8, 4))
for kind in ["gail", "airl", "logd"]:
    curve = np.asarray(reward_runs[kind]["env_ret"], dtype=np.float32)
    plt.plot(np.arange(1, len(curve) + 1), curve, label=kind)
plt.axhline(expert_score, ls="--", c="gray", label="expert")
plt.xlabel("GAIL iteration")
plt.ylabel("mean complete-episode env return")
plt.title("Surrogate-reward comparison (same data and interaction budget)")
plt.legend()
plt.tight_layout()
plt.show()
# ==========================================

### TODO 12 — how few demonstrations does GAIL need? *(4 pts)*

Ho & Ermon's headline claim is expert-sample efficiency. Test it: for `n_episodes` in $\{1, 3, 10\}$, collect a **fresh** demo set (`subsample=20`, `seed=500 + n_episodes` — note a single 500-step episode yields only **25 pairs**) and run GAIL (`kind="gail"`, 60 iters, no early stop, `verbose=False`, **~2 min each**). Store `(hist, final greedy eval)` in `demo_runs[n_episodes]`, then plot the learning curves together plus your 25-episode main run, and print a final-score table.

In [ ]:
demo_runs = {}
# ========== YOUR CODE HERE (~15 lines of solution) ==========
demo_pair_counts = {}
for n_ep in [1, 3, 10]:
    S_n, A_n, rets_n = collect_demos(
        expert, n_episodes=n_ep, subsample=20, seed=500 + n_ep
    )
    demo_pair_counts[n_ep] = len(S_n)
    ac_n, _, hist_n = gail(
        S_n, A_n, kind="gail", n_iters=60, target=None,
        seed=SEED, verbose=False
    )
    score_n = evaluate(greedy_fn(ac_n), n_episodes=20)
    demo_runs[n_ep] = (hist_n, score_n)
    print(f"collected {len(S_n)} retained pairs from {n_ep} complete episode(s); "
          f"mean demo return {np.mean(rets_n):.1f}")

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, len(gail_hist["env_ret"]) + 1), gail_hist["env_ret"],
         label=f"25 episodes ({len(demo_S)} pairs)")
for n_ep in [1, 3, 10]:
    hist_n, _ = demo_runs[n_ep]
    plt.plot(np.arange(1, len(hist_n["env_ret"]) + 1), hist_n["env_ret"],
             label=f"{n_ep} episode(s) ({demo_pair_counts[n_ep]} pairs)")
plt.axhline(expert_score, ls="--", c="gray", label="expert")
plt.xlabel("GAIL iteration")
plt.ylabel("mean complete-episode env return")
plt.title("Expert-demonstration sample study")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()
# ==========================================

for n_ep, (hist_n, score) in sorted(demo_runs.items()):
    print(f"{n_ep:2d} demo episode(s) ({demo_pair_counts[n_ep]:3d} pairs) -> "
          f"restored-best eval {score:6.1f}; final iterate "
          f"{hist_n['final_eval']:6.1f}; best iter {hist_n['best_iter']:2d}")

### TODO 13 — occupancy measures, visualized *(4 pts)*

GAIL's whole promise is $\rho_\pi \to \rho_{\pi^E}$. Look at it: implement `collect_states`, then draw three side-by-side scatter plots of the states visited by the **expert**, your **GAIL policy**, and a **random policy** (5000 greedy steps each), projected onto the two dimensions that matter for CartPole — pole angle $\theta$ (`obs[2]`) vs. angular velocity $\omega$ (`obs[3]`). Use a small marker size and low alpha, and share the axes so the supports are comparable. A successful GAIL run is visually indistinguishable from the expert; the random policy sprays over a much larger region.

In [ ]:
def collect_states(policy, n_steps=5000, seed=42):
    """Roll out `policy` (obs -> int action) for n_steps, resetting at episode ends.
    Returns float32 [n_steps, OBS_DIM] of visited states."""
    # ========== YOUR CODE HERE (~10 lines of solution) ==========
    env = gym.make(ENV_ID)
    states = np.zeros((n_steps, OBS_DIM), dtype=np.float32)
    episode = 0
    obs, _ = env.reset(seed=seed)
    for t in range(n_steps):
        states[t] = obs
        action = int(policy(obs))
        obs, _, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            episode += 1
            obs, _ = env.reset(seed=seed + episode)
    env.close()
    return states
    # ==========================================

# ========== YOUR CODE HERE (~10 lines of solution) ==========
_random_rng = np.random.default_rng(2026)
S_expert = collect_states(greedy_fn(expert), n_steps=5000, seed=42)
S_gail = collect_states(greedy_fn(gail_ac), n_steps=5000, seed=42)
S_random = collect_states(lambda obs: int(_random_rng.integers(N_ACTIONS)),
                          n_steps=5000, seed=42)

# Notebook-requested support view: identical axes and sampling convention.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, states, title in zip(
        axes, [S_expert, S_gail, S_random], ["expert", "GAIL", "random"]):
    ax.scatter(states[:, 2], states[:, 3], s=4, alpha=0.12)
    ax.set_title(title)
    ax.set_xlabel("pole angle θ")
axes[0].set_ylabel("angular velocity ω")
fig.suptitle("State-visitation support: θ versus ω")
plt.tight_layout()
plt.show()

# Implementation-box view: common bins; every empirical occupancy sums to one.
all_states = np.concatenate([S_expert, S_gail, S_random], axis=0)
theta_edges = np.linspace(all_states[:, 2].min(), all_states[:, 2].max(), 61)
omega_edges = np.linspace(all_states[:, 3].min(), all_states[:, 3].max(), 61)
occupancies = []
for states in [S_expert, S_gail, S_random]:
    H, _, _ = np.histogram2d(states[:, 2], states[:, 3],
                             bins=[theta_edges, omega_edges])
    H = H / H.sum()
    occupancies.append(H)
vmax = max(H.max() for H in occupancies)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharex=True, sharey=True)
for ax, H, title in zip(axes, occupancies, ["expert", "GAIL", "random"]):
    im = ax.imshow(H.T, origin="lower", aspect="auto",
                   extent=[theta_edges[0], theta_edges[-1],
                           omega_edges[0], omega_edges[-1]],
                   vmin=0.0, vmax=vmax)
    ax.set_title(f"{title} (mass={H.sum():.1f})")
    ax.set_xlabel("pole angle θ")
axes[0].set_ylabel("angular velocity ω")
fig.colorbar(im, ax=axes.ravel().tolist(), label="normalized visitation mass")
fig.suptitle("Normalized empirical state occupancy with common bins")
plt.show()
# ==========================================

assert S_expert.shape == (5000, OBS_DIM) and S_random.shape == (5000, OBS_DIM)
# the random policy's occupancy support must be much wider than the expert's
assert S_random[:, 2].std() > 2 * S_expert[:, 2].std(), "these do not look like the right policies"

### TODO 14 — what reward did the discriminator invent? *(4 pts)*

The trained discriminator *is* a learned reward function. Render it: on a grid of pole angle $\theta \in [-0.21, 0.21]$ × angular velocity $\omega \in [-2.5, 2.5]$ (cart position and velocity fixed at 0), evaluate the `gail` surrogate reward of your trained `gail_disc` for **each action**, and show the two heatmaps side by side (`imshow` with `extent=`, `origin="lower"`, a shared color scale, and a colorbar).

Then read it *carefully*, with your TODO 13 plot next to it. The expert only ever visits a sliver of this grid ($|\theta| \lesssim 0.05$, $|\omega| \lesssim 0.5$); everywhere else, both classes of training data are absent and the heatmap shows pure **extrapolation** — do not expect a physics-textbook control law out there, and don't be surprised if the extrapolated preferences even contradict physics. Two things to ponder (and to discuss in a sentence or two in this cell's output or a comment): (i) why does GAIL work anyway, given that its reward is only trustworthy on the expert's manifold? (Hint: where does the *policy* live by the end of training — and which direction does the reward push it when it strays?) (ii) Why does this make a GAIL discriminator, unlike AIRL's recovered reward, non-transferable to a new environment?

In [ ]:
thetas = np.linspace(-0.21, 0.21, 61)
omegas = np.linspace(-2.5, 2.5, 61)
# ========== YOUR CODE HERE (~18 lines of solution) ==========
Theta, Omega = np.meshgrid(thetas, omegas)
grid_states = np.zeros((Theta.size, OBS_DIM), dtype=np.float32)
grid_states[:, 2] = Theta.ravel().astype(np.float32)
grid_states[:, 3] = Omega.ravel().astype(np.float32)
reward_grids = []
for action in range(N_ACTIONS):
    grid_actions = np.full(Theta.size, action, dtype=np.int64)
    reward_grids.append(
        gail_reward(gail_disc, grid_states, grid_actions, kind="gail").reshape(Theta.shape)
    )

vmin = min(R.min() for R in reward_grids)
vmax = max(R.max() for R in reward_grids)
fig, axes = plt.subplots(1, N_ACTIONS, figsize=(10, 3.8), sharex=True, sharey=True)
for action, (ax, R) in enumerate(zip(np.atleast_1d(axes), reward_grids)):
    im = ax.imshow(R, extent=[thetas[0], thetas[-1], omegas[0], omegas[-1]],
                   origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_title(f"action {action}: {'left' if action == 0 else 'right'}")
    ax.set_xlabel("pole angle θ")
np.atleast_1d(axes)[0].set_ylabel("angular velocity ω")
fig.colorbar(im, ax=np.atleast_1d(axes).ravel().tolist(), label="GAIL surrogate reward")
fig.suptitle("Reward induced by the trained discriminator")
plt.show()

print("Interpretation: GAIL can succeed even though this reward is trustworthy mainly near the "
      "visited expert/policy support: on-policy updates keep collecting data where the policy lives, "
      "and deviations into policy-only regions are pushed back toward expert-like occupancy. Values "
      "far outside that support are unconstrained extrapolation. Because those extrapolated values "
      "depend on the original dynamics and visitation distribution rather than an identified task "
      "reward, the discriminator reward is not reliably transferable to a new environment.")
# ==========================================

### Final scoreboard *(given — run it)*

In [ ]:
results = {
    "expert (PPO on true reward)": expert_score,
    f"GAIL 'gail' best, {len(demo_S)} pairs": gail_score,
    "GAIL 'airl' best checkpoint": reward_scores["airl"],
    "GAIL 'logd' best checkpoint": reward_scores["logd"],
    f"GAIL best, 1 demo ({demo_pair_counts[1]} pairs)": demo_runs[1][1],
    f"GAIL best, 3 demos ({demo_pair_counts[3]} pairs)": demo_runs[3][1],
    f"GAIL best, 10 demos ({demo_pair_counts[10]} pairs)": demo_runs[10][1],
}
print("All GAIL scores below are restored-best checkpoints (20 fixed evaluation episodes).")
print("Final-iterate scores are reported in TODOs 11 and 12.")
for k, v in results.items():
    print(f"{k:42s} {v:6.1f}")

plt.figure(figsize=(7, 3.2))
names = list(results)
plt.barh(names[::-1], [results[n] for n in names[::-1]], color="tab:blue")
plt.axvline(500, ls="--", c="gray")
plt.xlabel("greedy eval return"); plt.title("HW7 — GAIL final comparison")
plt.tight_layout(); plt.show()

# Part 6 — Final Questions

## Q1 — The optimal discriminator

For each state-action pair, the optimal discriminator compares how often that pair appears in the expert data with how often it appears in the policy data. It gives a higher score when the pair is more common in the expert demonstrations and a lower score when it is more common in the policy rollouts. Substituting the optimal discriminator into the objective gives two times the Jensen–Shannon divergence minus log 4. Therefore, at the saddle point, the policy tries to minimize the Jensen–Shannon divergence between its state-action occupancy and the expert occupancy.

---

## Q2 — Reward bias

A strictly positive reward encourages survival because every extra step produces more reward, while reaching the terminal state gives no future reward. A strictly negative reward encourages early termination because ending the episode avoids collecting more negative rewards. This makes the positive `gail` reward helpful on CartPole because the goal is to keep the pole balanced for as long as possible. In our run, `gail` reached a restored-best score of **456.2**, while `airl` and `logd` scored about **9.3**, showing strong early-termination behavior. In an environment such as MountainCar, where reaching the goal ends the episode, a negative step reward can instead help the agent finish faster.

---

## Q3 — The discriminator must not win

If the discriminator becomes almost perfect while the policy is still far from the expert, it gives very negative logits to most policy samples. For very negative logits, the softplus reward and its gradient become close to zero, so the policy receives almost no useful information about which actions are better. Gradient penalty keeps the discriminator smoother, label smoothing prevents extreme confidence, and using only a few discriminator updates stops it from becoming too strong. Accuracy near 1.0 is usually a bad sign because the policy and expert are still easily separated and the reward may be saturated. Accuracy near 0.5 can also be bad if it happens from the beginning because the discriminator may be too weak, although it is expected near successful occupancy matching. Our final accuracies of about **0.71** for expert samples and **0.75** for policy samples show that the discriminator still had a useful but not perfect signal.

---

## Q4 — GAIL is not a GAN

A normal GAN can backpropagate the discriminator gradient directly through the generated sample, but GAIL cannot do this because actions are sampled and the environment transition is not differentiable. GAIL instead converts the discriminator output into a reward and uses an on-policy policy-gradient update with PPO. This replacement has higher variance because it depends on sampled actions and sampled trajectories. The value baseline, GAE, advantage normalization, PPO clipping, minibatches, and gradient clipping reduce this variance and improve stability. The code hyperparameter that plays the role of the causal-entropy coefficient is `ent_coef`. Setting it to zero removes the exploration bonus, so the policy may become deterministic too early and get stuck in a weaker behavior.

---

## Q5 — Two sample complexities

GAIL was very efficient in its use of expert demonstrations in our run. With only **25 expert state-action pairs**, it reached a best score of **500.0**. With **75 pairs**, it reached **326.3**, and with **243 pairs**, it again reached **500.0**. A behavioral-cloning model trained on only 25 pairs would normally have poor coverage and would not know what to do after reaching states outside the small dataset. GAIL gets additional information by collecting fresh environment rollouts and learning from the states visited by its own policy. Matching the full state-action occupancy helps reduce behavioral cloning’s compounding-error problem because mistakes and unfamiliar states appear in the policy data and influence later updates. However, GAIL needs many environment interactions, so behavioral cloning is still preferable when interaction is expensive, dangerous, slow, or unavailable, such as learning from offline human-driving data.


---

# Part 7 — Theoretical Questions

## Question 4 — From Occupancy Matching to GAIL

### 1.

The optimal discriminator compares the expert occupancy with the policy occupancy at each state-action pair. It gives a high value when a pair is more common in the expert data and a low value when it is more common in policy data. After substituting this optimal discriminator into the adversarial objective, the result is two times the Jensen–Shannon divergence between the expert and policy occupancies, minus log 4.

### 2.

At the saddle point, the policy minimizes the Jensen–Shannon divergence between its state-action occupancy and the expert occupancy. This is stronger than behavioral cloning because behavioral cloning only learns the expert action on states contained in the demonstration dataset. A small mistake can move the cloned policy into an unseen state, where further mistakes can grow over time. GAIL compares complete state-action visitation distributions, so states reached because of policy mistakes also appear in training and the policy receives pressure to move back toward expert-like behavior.

### 3.

GAIL cannot backpropagate the discriminator gradient through a sampled environment transition because the sampled action and environment dynamics are not differentiable. Instead, the discriminator output is treated as a reward, and the policy is updated using an on-policy policy-gradient estimator with PPO. The value baseline removes part of the variance, GAE balances bias and variance when estimating advantages, advantage normalization keeps update scales stable, and PPO clipping prevents excessively large policy changes.

### 4.

The code hyperparameter corresponding to the causal-entropy coefficient is `ent_coef`. It rewards policy entropy and encourages exploration. Setting it to zero removes this exploration encouragement, which may make the policy become deterministic too early, reduce action diversity, and cause convergence to a poor local solution.

---

## Question 5 — Reward Bias and Adversarial Training Dynamics

### 1.

An always-positive reward favors long episodes because every additional step gives more reward, while termination gives no continuation value. An always-negative reward favors early termination because stopping avoids future negative rewards. On CartPole, the positive bias is helpful because the true goal is to survive for as many steps as possible. This matches our results: the positive `gail` reward reached a restored-best score of **456.2**, while `airl` and `logd` scored about **9.3** and often ended episodes quickly. The preference can reverse in MountainCar, where the episode ends after reaching the goal and a negative step cost encourages faster completion.

### 2.

A nearly perfect discriminator can destroy the policy’s learning signal when the policy is far from the expert. It produces saturated negative logits for policy samples, and the softplus reward then becomes very small and changes very little between actions. The policy therefore cannot clearly tell which behavior is moving toward the expert. Gradient penalty limits sharp discriminator changes, label smoothing reduces extreme confidence, and a small number of discriminator updates keeps the discriminator from moving too far ahead. Accuracy near 1.0 usually means the discriminator is too strong and the occupancies remain easily separable. Accuracy near 0.5 is expected near successful matching, but if it appears from the beginning with no improvement, it may mean the discriminator is too weak or has collapsed. Our final accuracies around **0.71** and **0.75** show useful separation without a fully perfect discriminator.

### 3.

GAIL may need very few expert demonstrations because it uses the expert data repeatedly while collecting many new policy transitions from the environment. In our results, only **25 retained expert pairs** were enough to produce a best score of **500.0**, although the **75-pair** run reached only **326.3**, showing that adversarial training has high variance. Expert-sample complexity can therefore be low, but environment-interaction complexity remains high because every training iteration requires fresh on-policy rollouts. This makes GAIL impractical in settings where real-world interaction is expensive, slow, unsafe, or impossible, even when expert demonstrations are limited.